In [2]:
import pandas as pd
import numpy as np
import sklearn

In [186]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.0/461.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 23.8 MB/s eta 0:00:00
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.2
    Uninstalling tzdata-2026.2:
      Successfully uninstalled tzdata-2026.2
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: narwhals
    Found e

In [3]:
print("np",np.__version__)
print("pd",pd.__version__)
print("sklearn",sklearn.__version__)

np 2.5.1
pd 3.0.3
sklearn 1.9.0


## Preparing training and testing dataset

In [4]:
from sklearn.datasets import load_iris

In [5]:
df=load_iris(as_frame=True)

In [6]:
df=df.frame

In [73]:
df.sample(5)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
32,5.2,4.1,1.5,0.1,0
78,6.0,2.9,4.5,1.5,1
58,6.6,2.9,4.6,1.3,1
89,5.5,2.5,4.0,1.3,1
65,6.7,3.1,4.4,1.4,1


In [8]:
df.shape

(150, 5)

In [9]:
df.isnull().sum()

sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

In [33]:
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=4)

In [34]:
from sklearn.preprocessing import OneHotEncoder

In [35]:
OHE=OneHotEncoder()

In [36]:
y_train=OHE.fit_transform(y_train.values.reshape(-1, 1))


##Model

In [15]:
np.random.seed(42)

In [37]:
class Softmax_regression():
  def __init__(self,learning_rate,epoch,batch_sz,num_class):
    self.weights=None
    self.lr=learning_rate
    self.epochs=epoch
    self.batch_size=batch_sz
    self.classes=num_class

  def softmax(self,z):
    # Ensure z is an ndarray to avoid issues with numpy.matrix's sum method
    z = np.asarray(z)
    expZ=np.exp(z)
    return expZ/np.sum(expZ,axis=1,keepdims=True)

  def fit(self,X_train,y_train):
    X_train=np.insert(X_train,0,1,axis=1)
    self.weights=np.ones((self.classes,X_train.shape[1]))

    for i in range(self.epochs):

      idxs = np.random.randint(0, X_train.shape[0]-1, self.batch_size)
      x_train=X_train[idxs]
      z=np.dot(X_train[idxs],self.weights.T)
      y_pred=self.softmax(z)

      #weights gradient
      weights_grad=np.dot((y_pred-y_train[idxs]).T,x_train)
      #updation
      self.weights=self.weights-self.lr*weights_grad

  def predict(self,x_test):
      x_test=np.insert(x_test,0,1,axis=1)
      return np.argmax(np.dot(x_test,self.weights.T),axis=1)

##Model.fit()

In [62]:
model=Softmax_regression(0.1,250,15,3)

In [63]:
model.fit(X_train,y_train)

In [64]:
y_got=model.predict(X_test)

##Accuracy

In [52]:
from sklearn.metrics import accuracy_score

In [65]:
print(f"{np.round(accuracy_score(np.asarray(y_test.values), np.asarray(y_got))*100)}%")

97.0%


##Classification Report

In [66]:
from sklearn.metrics import classification_report

In [67]:
print(classification_report(np.asarray(y_test.values),np.asarray(y_got)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       1.00      0.80      0.89         5
           2       0.90      1.00      0.95         9

    accuracy                           0.97        30
   macro avg       0.97      0.93      0.95        30
weighted avg       0.97      0.97      0.97        30



##pkl file

In [74]:
import joblib

In [82]:
target_names = ['setosa', 'versicolor', 'virginica']
model_data = {
    'model': model,
    'target_names': list(target_names) # ['setosa', 'versicolor', 'virginica']
}

In [83]:
joblib.dump(model_data, 'model.pkl')
print("Model saved perfectly using standard Joblib!")

Model saved perfectly using standard Joblib!
